In [1]:
from transformers import AutoProcessor, AutoModelForCausalLM
from qwen_vl_utils import process_vision_info  # you can still use this if it just extracts PIL images
import torch
from pathlib import Path
import pandas as pd
import ast
from PIL import Image, ImageDraw, ImageFont
import math

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
COLOR_MAPPING = {
    (0xFF, 0xD7, 0x00): 'common room',
    (0xFF, 0xA5, 0x00): 'master room',
    (0xEE, 0xE8, 0xAA): 'living room',
    (0x6B, 0x8E, 0x23): 'balcony',
    (0xAD, 0xD8, 0xE6): 'bathroom',
    (0xF0, 0x80, 0x80): 'kitchen',
    (0xDD, 0xA0, 0xDD): 'storage',
    (0xDA, 0x70, 0xD6): 'dining',
}

ANNOT_DIR = Path("../../annotation/human_annotated_tags")
DATA_DIR = Path("../../data/floorplan_image")

In [3]:
# class LocalVisionLLM:
#     def __init__(self, model_id: str, device: str = 'cuda'):
#         # Processor for both vision & text
#         self.processor = AutoProcessor.from_pretrained(
#             model_id,
#             use_auth_token=True,
#             trust_remote_code=True
#         )

#         # Vision‑language conditional generation model
#         self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#             model_id,
#             torch_dtype=torch.float16,
#             device_map='auto',
#             trust_remote_code=True,
#             use_auth_token=True
#         )
#         # track actual device (could be spread across GPUs)
#         self.device = next(self.model.parameters()).device

#     def __call__(self, images: list[Image.Image], prompt: str, **generate_kwargs) -> str:
#         # 1) Build the “chat” messages list
#         messages = [{"type": "text",  "content": prompt}]
#         messages += [{"type": "image", "content": img} for img in images]

#         # 2) Turn messages into a single text string with the model’s chat template
#         #    (this adds any necessary <|user|>, <|assistant|> tokens and the generation prompt)
#         chat_text = self.processor.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=True
#         )

#         # 3) Extract visual inputs (pixel buffers, bboxes, etc.)
#         image_inputs, video_inputs = process_vision_info(messages)  # returns two things; video_inputs will be None here :contentReference[oaicite:0]{index=0}

#         # 4) Tokenize the chat text
#         text_inputs = self.processor(
#             chat_text,
#             return_tensors="pt",
#             add_special_tokens=False  # tokens already handled by apply_chat_template
#         )

#         # 5) Merge text + images into one input dict
#         #    (drop video_inputs since you’re only doing stills)
#         inputs = {**text_inputs, **(image_inputs or {})}
#         inputs = {k: v.to(self.device) for k, v in inputs.items()}

#         # 6) Generate
#         defaults = dict(max_new_tokens=256, do_sample=False)
#         outputs = self.model.generate(**inputs, **{**defaults, **generate_kwargs})

#         # 7) Decode & strip off the echoed prompt
#         full = self.processor.decode(outputs[0], skip_special_tokens=True)
#         return full[len(prompt):].strip()

In [3]:
class LocalVisionLLM:
    def __init__(self, model_id: str = "microsoft/Phi-4-multimodal-instruct",
                 device_map: str = 'auto', torch_dtype="auto"):
        # Load processor & model
        self.processor = AutoProcessor.from_pretrained(
            model_id,
            trust_remote_code=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            device_map=device_map,
            trust_remote_code=True
        )
        self.device = next(self.model.parameters()).device

        # Phi‑4 chat delimiters
        self.user_tok  = "<|user|>"
        self.assist_tok= "<|assistant|>"
        self.end_tok   = "<|end|>"
        self.image_tok = "<|image_1|>"


    def __call__(self, images: list[Image.Image], prompt: str,
                 max_new_tokens: int = 256, **gen_kwargs) -> str:
        # Build one tag per image
        image_tags = "".join(f"<|image_{i+1}|>" for i in range(len(images)))
        full_prompt = (
            f"{self.user_tok}"
            f"{image_tags}"
            f"{prompt}"
            f"{self.end_tok}"
            f"{self.assist_tok}"
        )

        # Prepare inputs
        inputs = self.processor(
            text=full_prompt,
            images=images,
            return_tensors="pt",
            padding=True
        ).to(self.device)

        # Fix generation config to avoid num_logits_to_keep=None
        generation_config = {
            "max_new_tokens": max_new_tokens,
            "num_beams": 1,
            "num_logits_to_keep": 1,  # Explicitly set this to avoid TypeError
            "use_cache": True,
            **gen_kwargs
        }

        # Generate
        with torch.no_grad():
            output_ids = self.model.generate(**inputs, **generation_config)

        # Strip prompt, decode
        input_len = inputs["input_ids"].shape[1]
        reply_ids = output_ids[:, input_len:]
        reply = self.processor.batch_decode(
            reply_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0].strip()

        return reply


In [4]:
model_id = "microsoft/Phi-4-multimodal-instruct"
vlm = LocalVisionLLM(model_id)

/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/models/auto/image_processing_auto.py:604: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/hice1/hzhang931/.cache/huggingface/modules/transformers_modules/microsoft/Phi-4-multimodal-instruct/33e62acdd07cd7d6635badd529aa0a3467bb9c6a/speech_conformer_encoder.py:2774: FutureWarning: Please specify CheckpointImpl.NO_REENTRANT as CheckpointImpl.REENTRANT will soon be removed as the default and eventually deprecated.
  lambda i: encoder_check

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
csv_path = Path("groups_same_second_results.csv")
# csv_path = Path("groups_diff_first_results.csv")
# csv_path = Path("groups_same_first_global_results.csv")
df = pd.read_csv(csv_path)

In [6]:
# def build_prompt(options: list[str]) -> str:
#     text = (
#         "I am showing you 5 apartment floorplan images.\n"
#         "4 out of these descriptions share the same underlying floorplan pattern and 1 out of them has a different floorplan pattern\n\n"
#         "Please look carefully at spatial relationships, room types, and sizes.\n"
#         "Which example is the different one, and why?\n\n"
#         "Structure your reply exactly like:\n"
#         "1. **Different example:** <ID>\n"
#         "2. **Why:** <brief reasoning>\n\n"
#         "Examples by number:\n"
#     )
#     for idx, pid in enumerate(options, start=1):
#         text += f"- Example {idx}: ID {pid}\n"
#     return text

# def build_prompt(options: list[str]) -> str:
#     text = (
#         "I am showing you 5 apartment floorplan images.\n"
#         "4 out of these descriptions share the same underlying floorplan pattern and 1 out of them has a different floorplan pattern\n\n"
#         "Please look carefully at spatial relationships, room types, and sizes.\n"
#         "\n"
#         "Examples by number:\n"
#     )
#     for idx, pid in enumerate(options, start=1):
#         text += f"- Example {idx}: ID {pid}\n"

#     text += (
#         "\nWhich example is the different one, and why?\n\n"
#         "Structure your reply exactly like:\n"
#         "1. **Different example:** <ID>\n"
#         "2. **Why:** <brief reasoning>\n\n"
#     )

#     return text

# def build_prompt(options: list[str]) -> str:
#     text = (
#         "I am showing you 5 apartment floorplan images.\n"
#         "4 out of these descriptions share the same underlying floorplan pattern and 1 out of them has a different floorplan pattern\n\n"
#         "Please look carefully at spatial relationships, room types, and sizes.\n"
#         "Which example is the different one, and why?\n\n"
#         "Structure your reply exactly like:\n"
#         "1. **Different example:** <ID>\n"
#         "2. **Why:** <brief reasoning>\n\n"
#         "Examples by number:\n"
#     )
#     for idx, pid in enumerate(options, start=1):
#         text += f"- Example {idx}: ID {pid}\n"
#     return text

def build_prompt(options: list[str]) -> str:
    text = (
        "I am showing you 5 apartment floorplan images.\n"
        "4 out of these descriptions share the same underlying floorplan pattern and 1 out of them has a different floorplan pattern\n\n"
        "Each of these examples come with a legend indicating the correspondence between colors and room types.\n"
        "Please look carefully at spatial relationships, room types, and sizes.\n"
        "\n"
        "Examples by number:\n"
    )
    for idx, pid in enumerate(options, start=1):
        text += f"- Example {idx}: ID {pid}\n"

    text += (
        "\nWhich example is the different one, and why?\n\n"
        "Structure your reply exactly like:\n"
        "1. **Different example:** <ID>\n"
        "2. **Why:** <brief reasoning>\n\n"
    )

    return text

def load_annotation(pid: str) -> str:
    """
    Load the human-annotated textual description for a given floorplan ID.
    """
    txt_path = ANNOT_DIR / f"{pid}.txt"
    try:
        return txt_path.read_text(encoding='utf-8').strip()
    except FileNotFoundError:
        return "<No annotation available>"

def build_2modal_prompt(options: list[str]) -> str:
    text = (
        "I am showing you 5 apartment floorplan images, each with a corresponding human-annotated description.\n"
        "4 out of these examples share the same underlying floorplan pattern and 1 is an outlier.\n"
        "Please examine the spatial relationships, room types, sizes, and provided descriptions.\n\n"
        "Examples by number:\n"
    )

    for idx, pid in enumerate(options, start=1):
        # load textual annotation
        annotation = load_annotation(pid)
        text += f"- Example {idx}: ID {pid}\n"
        text += f"  Description: {annotation}\n"

    text += (
        "\nWhich example is the different one, and why?\n\n"
        "Structure your reply exactly like:\n"
        "1. **Different example:** <ID>\n"
        "2. **Why:** <brief reasoning>\n\n"
    )

    # print(text)

    return text


In [ ]:
# # 2. iterate
# results = []
# data_dir = Path("../../data/floorplan_reoriented")

# for _, row in df.iterrows():
#     options = ast.literal_eval(row['options'])
#     prompt  = build_prompt(options)

#     # load the five floorplan images
#     images = []
#     for pid in options:
#         img_path = data_dir / f"{pid}.png"
#         images.append(Image.open(img_path).convert("RGB"))

#     # call your vision‑language LLM
#     # it handles processor, device placement, process_vision_info, generate(), decode(), and prompt‑stripping

#     answer = vl_llm(images, prompt, max_new_tokens=256, do_sample=False)

#     # parse out the “1. Different example: X” line
#     first_line = answer.splitlines()[0]
#     predicted = first_line.split(":", 1)[1].strip()

#     results.append({
#         'base_cluster':  row['base_cluster'],
#         'other_cluster': row['other_cluster'],
#         'options':       options,
#         'outlier_id':    row['outlier_id'],
#         'predicted_id':  predicted,
#         'raw_response':  answer
#     })

# # 3. save
# df_out = pd.DataFrame(results)
# df_out.to_csv("outputs/llm_image_results_qwen2.5-vl.csv", index=False)
# print(f"Wrote {len(df_out)} results to outputs/llm_image_results_qwen2.5-vl.csv")

TypeError: 'Image' object is not iterable

In [9]:
results = []
data_dir = Path("../../data/floorplan_image")

for _, row in df.iterrows():
    options = ast.literal_eval(row['options'])
    prompt  = build_prompt(options)

    images = [
        Image.open(data_dir / f"{pid}.png").convert("RGB")
        for pid in options
    ]

    answer = vlm(images, prompt, do_sample=False)

    first_line = answer.splitlines()[0]
    predicted = first_line.split(":", 1)[1].strip()

    results.append({
        'base_cluster':  row['base_cluster'],
        'other_cluster': row['other_cluster'],
        'options':       options,
        'outlier_id':    row['outlier_id'],
        'predicted_id':  predicted,
        'raw_response':  answer
    })

df_out = pd.DataFrame(results)
out_path = Path("outputs/vlm_phi4_multimodal_same_second.csv")
out_path.parent.mkdir(exist_ok=True)
df_out.to_csv(out_path, index=False)
print(f"Wrote {len(df_out)} results to {out_path}")

Wrote 100 results to outputs/vlm_phi4_multimodal_same_second.csv


In [7]:
def add_legend_two_columns(image: Image.Image, mapping: dict) -> Image.Image:
    swatch = 20
    pad    = 5
    font   = ImageFont.load_default()

    entries = list(mapping.items())
    cols    = 2
    rows    = math.ceil(len(entries) / cols)
 
    leg_h = rows * (swatch + pad) + pad
    leg_w = image.width 

    legend = Image.new("RGB", (leg_w, leg_h), "white")
    draw   = ImageDraw.Draw(legend)

    col_w = leg_w // cols

    for idx, (color, label) in enumerate(entries):
        col = idx // rows
        row = idx % rows
        x0  = col*col_w + pad
        y0  = row*(swatch + pad) + pad

        draw.rectangle([x0, y0, x0+swatch, y0+swatch], fill=color)
        draw.text((x0+swatch+pad, y0), label, fill="black", font=font)

    combined = Image.new("RGB", (image.width, image.height + leg_h))
    combined.paste(image, (0, 0))
    combined.paste(legend, (0, image.height))
    return combined

In [23]:
results = []
data_dir = Path("../../data/floorplan_image")

for _, row in df.iterrows():
    options = ast.literal_eval(row['options'])
    prompt  = build_prompt(options)

    images = []

    for pid in options:
        img = Image.open(data_dir / f"{pid}.png").convert("RGB")
        img = add_legend_two_columns(img, COLOR_MAPPING)
        images.append(img)

    answer = vlm(images, prompt, do_sample=False)

    first_line = answer.splitlines()[0]
    predicted = first_line.split(":", 1)[1].strip()

    results.append({
        'base_cluster':  row['base_cluster'],
        'other_cluster': row['other_cluster'],
        'options':       options,
        'outlier_id':    row['outlier_id'],
        'predicted_id':  predicted,
        'raw_response':  answer
    })

df_out = pd.DataFrame(results)
out_path = Path("outputs/vlm_phi4_multimodal_same_first_global_wlegend.csv")
out_path.parent.mkdir(exist_ok=True)
df_out.to_csv(out_path, index=False)
print(f"Wrote {len(df_out)} results to {out_path}")

Wrote 100 results to outputs/vlm_phi4_multimodal_same_first_global_wlegend.csv


In [ ]:
results = []
data_dir = Path("../../data/floorplan_image")

for _, row in df.iterrows():
    options = ast.literal_eval(row['options'])
    prompt  = build_2modal_prompt(options)

    images = []

    for pid in options:
        img = Image.open(DATA_DIR / f"{pid}.png").convert("RGB")
        img = add_legend_two_columns(img, COLOR_MAPPING)
        images.append(img)

    answer = vlm(images, prompt, do_sample=False)

    first_line = answer.splitlines()[0]
    predicted = first_line.split(":", 1)[1].strip()

    results.append({
        'base_cluster':  row['base_cluster'],
        'other_cluster': row['other_cluster'],
        'options':       options,
        'outlier_id':    row['outlier_id'],
        'predicted_id':  predicted,
        'raw_response':  answer
    })

df_out = pd.DataFrame(results)
out_path = Path("outputs/vlm_phi4_multimodal_2modal_same_second_wlegend.csv")
out_path.parent.mkdir(exist_ok=True)
df_out.to_csv(out_path, index=False)
print(f"Wrote {len(df_out)} results to {out_path}")

If you are not using the generate method, you may encounter nonsensical outputs after the 4096th token, as the KV cache needs to be recomputed.
We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)
If you are not using the generate method, you may encounter nonsensical outputs after the 4096th token, as the KV cache needs to be recomputed.
If you are not using the generate method, you may encounter nonsensical outputs after the 4096th token, as the KV cache needs to be recomputed.
If you are not using the generate method, you may encounter nonsensical outputs after the 4096th token, as the KV cache needs to be recomputed.
If you are not using the generate method, you may encounter nonsensical outputs after the 4096th token, as the KV cache needs to be recomputed.
If you are not using the

Wrote 100 results to outputs/vlm_phi4_multimodal_2modal_same_second_wlegend.csv
